# Project 02 — Estimating a Mean & Spread (Normal $\mu,\sigma$)

**Scenario.** A concentration is measured $N$ times on the same sample. Each replicate is the true concentration $\mu$ plus Gaussian noise of unknown SD $\sigma$. We want the **joint** posterior for $(\mu,\sigma)$.

**New skill:** jointly inferring a *scale* $\sigma$ alongside a location $\mu$, with priors on both. **Key pitfall:** improper/uninformative scale priors — a flat prior on $\sigma$ is *not* harmless.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
az.style.use('arviz-darkgrid')
RNG = 20240602

## Step 1 — Problem & data-generating story

We assume each measurement is independent and Normally distributed around a common true concentration $\mu$ with constant noise SD $\sigma$. **Assumptions made explicit:** (a) measurements independent, (b) $\mu$ and $\sigma$ constant across replicates (no drift/heteroscedasticity), (c) noise is symmetric/Gaussian (no heavy tails or outliers). We synthesize from known $\mu_\text{true}=5.0$, $\sigma_\text{true}=1.2$.

In [ ]:
from data.generate_data import generate
data = generate()
y = data['y']
print(f"n={data['n']}, empirical mean={y.mean():.3f}, "
      f"empirical sd={y.std(ddof=1):.3f}")
print('true:', data['truth'])

## Step 2 — Model specification (likelihood + justified priors)

$$y_i \sim \text{Normal}(\mu, \sigma), \quad \mu \sim \text{Normal}(5, 10), \quad \sigma \sim \text{HalfNormal}(5).$$

**Why these priors?** The location prior $\text{Normal}(5,10)$ is *broad* (2 SD spans roughly $-15$ to $25$) — weakly-informative on the data's scale. The scale prior is where care matters: $\sigma$ must be positive, so we use a **proper** $\text{HalfNormal}(5)$, which concentrates mass on plausible noise levels and has a finite integral. A 'flat' improper prior on $\sigma$ (uniform on $(0,\infty)$) is **not** uninformative — it puts unbounded mass on absurdly large variances and can destabilize sampling. That is the seeded bug in the broken notebook.

In [ ]:
from model import build_model, fit
model = build_model(data)
model

## Step 3 — Prior predictive checks

We simulate datasets implied by the prior. With $\mu\sim N(5,10)$ and $\sigma\sim\text{HalfNormal}(5)$ the implied measurements should span a wide-but-not-insane range. If a flat $\sigma$ prior were used, the prior predictive would contain datasets with astronomically large spread — the visual tell of an improper scale prior.

In [ ]:
with model:
    prior = pm.sample_prior_predictive(draws=500, random_seed=RNG)
obs_dim = [d for d in prior.prior_predictive['y'].dims if d not in ('chain','draw')][0]
pp_means = prior.prior_predictive['y'].mean(dim=obs_dim).values.ravel()
pp_sds = prior.prior_predictive['y'].std(dim=obs_dim).values.ravel()
fig, axes = plt.subplots(1, 2, figsize=(9,3.5))
axes[0].hist(pp_means, bins=30, color='#55A868', edgecolor='white')
axes[0].set(xlabel='dataset mean implied by prior', ylabel='count', title='prior predictive means')
axes[1].hist(pp_sds, bins=30, color='#C44E52', edgecolor='white')
axes[1].set(xlabel='dataset sd implied by prior', ylabel='count', title='prior predictive spreads')
plt.tight_layout()

## Step 4 — Inference (NUTS)

We sample with `draws=1000, tune=1000, chains=4`. Four chains give reliable split-$\hat R$; ample tuning lets NUTS adapt its step size and mass matrix to the joint $(\mu,\sigma)$ geometry. The seed is fixed for reproducibility.

In [ ]:
idata = fit(data, draws=1000, tune=1000, chains=4, seed=202)

## Step 5 — Computational diagnostics

We check $\hat R\approx 1.00$, bulk/tail ESS $\gtrsim 400$ for **both** parameters, and 0 divergences. The trace should show well-mixed caterpillars. Scale parameters can be harder to sample than locations, so watch $\sigma$'s ESS in particular.

In [ ]:
print(az.summary(idata, var_names=['mu', 'sigma']))
n_div = int(idata.sample_stats['diverging'].sum())
print(f'divergences: {n_div}')

In [ ]:
az.plot_trace(idata, var_names=['mu', 'sigma']); plt.tight_layout()

**What if diagnostics fail?** Low ESS on $\sigma$ or divergences usually signal a bad scale prior or a sampler starved of tuning. Remedies: raise `target_accept`, increase `tune`, and — crucially — use a *proper* scale prior. An improper flat $\sigma$ prior often manifests as poor mixing and a posterior with a heavy right tail that never settles.

## Step 6 — Posterior predictive checks

We overlay replicated datasets from the posterior on the observed data. If the Normal model is adequate, the observed histogram/density sits inside the posterior-predictive band. This is also where non-Gaussian features (outliers, skew) would betray themselves — motivation for the robust models later.

In [ ]:
az.plot_ppc(idata, num_pp_samples=100); plt.tight_layout()

In [ ]:
pp = idata.posterior_predictive['y']
obs_dim = [d for d in pp.dims if d not in ('chain','draw')][0]
pp_sd = pp.std(dim=obs_dim).values.ravel()
obs_sd = y.std(ddof=1)
p_value = float(np.mean(pp_sd >= obs_sd))
print(f'observed sd={obs_sd:.3f}; posterior-predictive p-value for sd={p_value:.3f} (near 0.5 = good fit)')

## Step 7 — Model criticism & comparison

With a single model there is no LOO/WAIC comparison yet, but we criticize the fit two ways: (1) confirm the posterior for $(\mu,\sigma)$ brackets the known truth, and (2) inspect the joint posterior for the classic mild correlation between $\mu$ and $\sigma$ when $N$ is modest.

In [ ]:
az.plot_pair(idata, var_names=['mu', 'sigma'], kind='kde',
             marginals=True, figsize=(6,5))
plt.tight_layout()

## Step 8 — Decision & communication

Translate the joint posterior into reportable quantities: the estimated concentration with a credible interval, and the estimated measurement noise.

In [ ]:
mu_post = idata.posterior['mu'].values.ravel()
sigma_post = idata.posterior['sigma'].values.ravel()
mu_lo, mu_hi = np.percentile(mu_post, [3, 97])
sig_lo, sig_hi = np.percentile(sigma_post, [3, 97])
print(f'Concentration mu = {mu_post.mean():.3f} mg/mL, 94% CI [{mu_lo:.3f}, {mu_hi:.3f}]')
print(f'Measurement noise sigma = {sigma_post.mean():.3f} mg/mL, 94% CI [{sig_lo:.3f}, {sig_hi:.3f}]')

**Conclusion (for a collaborator).** The sample's concentration is about 5.0 mg/mL (94% CI roughly [4.6, 5.4]) and the assay's replicate noise is about 1.2 mg/mL. Reporting $\sigma$ matters as much as $\mu$: it tells the collaborator how reproducible a single future measurement will be.